Detection 과 Segmentation

YOLO V11n-seg

1. 바운딩박스(x1,y1,x2,y2, conf, class)
2. 마스크 >> (0, 1) 픽셀단위

In [1]:
# 1. Ultralytics 패키지 설치
!pip install ultralytics tqdm opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 788.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.4 MB/s eta 0:00:00


In [2]:
import os
import cv2
import torch
import numpy as np
import urllib.request
from tqdm import tqdm
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
video_url = "https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4"
video_path = "car_sample.mp4"
output_path = "lane_output_segmented.mp4"

urllib.request.urlretrieve(video_url, video_path)

('car_sample.mp4', <http.client.HTTPMessage at 0x7a3a8f95c8f0>)

In [6]:
model = YOLO("yolo11n-seg.pt")

In [8]:
# 비디오 캡처 객체 및 메타 데이터 정보 가져오기
cap = cv2.VideoCapture(video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# 영상이 제대로 안 받아졌을 경우를 대비한 안전 장치
if total_frames <= 0:
    raise ValueError("비디오 프레임이 0입니다. 영상이 제대로 다운로드되지 않았습니다.")

# 영상저장 설정: 결과 저장을 위한 VideoWriter 설정(mp4v codec사용)
# fourcc : 비디오 압축방식 >> mp4 형식(mp4v, XCID, MJPG,H264)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# VideoWriter 생성 (처리된 영상 저장할 객체 생성)
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

print(f"차선 검출 세그멘테이션 시작 (총 프레임 수: {total_frames})")

차선 검출 세그멘테이션 시작 (총 프레임 수: 377)


In [9]:
# 프레임별 실행
with tqdm(total=total_frames, desc="Processing Frames") as pbar:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # YOLO11-Seg 모델 예측 수행
        # model.track() : 객체 추적(object tracking)
        # persist=True (객체 ID 유지) 예) 자동차 A(ID_1), 자동차 B(ID_2)
        results = model.track(frame, persist=True, verbose=False)
        overlay = frame.copy() # 원본 복사(손상 방지용)
        result = results[0]    # 첫번째 결과물 가져오기

        # 세그멘테이션 마스크(Masks) 데이터가 존재하는지 확인
        # >> segmentation 결과가 있는 경우만 수행
        if result.masks is not None:
            # 바운딩 박스 추출
            masks = result.masks.data.cpu().numpy()
            boxes = result.boxes.data.cpu().numpy()
            # [x1, y1, x2, y2, conf, class]

            for mask, box in zip(masks, boxes):
                class_id = int(box[-1])  # box의 마지막 값(class)
                mask_resized = cv2.resize(mask, (width, height))
                # mask 크기 조정 (원본 영상에 맞춤)

                # 시각화 색상 지정 (2: 자동차, 7: 트럭, 0: 사람)
                if class_id in [2, 7]:
                    color = (255, 0, 0)     # 파란색 (B)
                elif class_id == 0:
                    color = (255, 0, 255)   # 보라색
                else:
                    color = (0, 255, 0)     # 초록색 (G)

                overlay[mask_resized > 0.5] = color
                # mask 값 범위 [0,1] >> 0.5 이상이면, 객체 영역
                # 자동차 부분 >> 파란색, 사람 부분 보라색

        # 6. 원본 프레임과 마스크 합성 (불투명도 40%)
        alpha = 0.4   # 투명도 설정
        output_frame = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)
        # dst = src1 * alpha + src2 * beta(1-alpha) + gamma
        # 40% 색칠, 60% 원본
        # overlay (src1): 세그멘테이션 색상(파란색, 보라색 등)이 칠해진 이미지
        # alpha : overlay 이미지의 가중치(0.4)
        # frame (src2): 원본 이미지
        # 1-alpha : 0.6
        # 0 gamma): 감마 값입니다. 전체적인 사진의 밝기를 강제로 올리고 싶을 때 사용
        # >> 보통 0을 주어 밝기 변화 없이 순수하게 두 이미지만 섞습니다.
        # 요약하자면?"원본 영상(frame)은 60% 진하게, 색칠된 마스크(overlay)는 40% 연하게 섞어서
        # 두 개를 겹친 새로운 픽셀을 만들어라."이 과정을 통해 차선이나 사람을 가리지 않고 그 위에 자연스러운 형광펜 효과

        out.write(output_frame)
        pbar.update(1)

# 자원 해제
cap.release()
out.release()
print(f"결과 영상({output_path})이 완성되었습니다.")

# 7. Google Colab 환경일 경우 결과 파일 자동 다운로드
try:
    from google.colab import files
    print("결과 영상을 내 컴퓨터로 다운로드합니다...")
    files.download(output_path)
except ImportError:
    print(f"현재 폴더에서 '{output_path}' 파일을 직접 확인해주세요!")

Processing Frames:   0%|          | 0/377 [00:00<?, ?it/s]

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 238ms
Prepared 1 package in 38ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



Processing Frames: 100%|██████████| 377/377 [01:18<00:00,  4.78it/s]

결과 영상(lane_output_segmented.mp4)이 완성되었습니다.
결과 영상을 내 컴퓨터로 다운로드합니다...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>